# Kaggle runner — train + evaluate one experiment

This notebook contains **no model code**. It clones the repo and calls the package, so every run uses
exactly the same code and the same evaluation protocol.

**Setup:** GPU T4 · Internet ON · Add Data → `andrewmvd/retinal-disease-classification`.
Set `EXPERIMENT` and `SEEDS` below, then *Save Version → Save & Run All*.

Budget (T4, 30 epochs): euclidean ≈ 1.5–2 h / seed · poincare ≈ 3 h / seed · lorentz ≈ 3 h / seed.
Kaggle sessions stop at 12 h, so keep ≤ 3 hyperbolic seeds per version.

In [ ]:
# ---- What to run ----
EXPERIMENT = "lorentz"          # euclidean | poincare | lorentz | smoke_test
SEEDS = [0]                     # e.g. [0, 1, 2]
REPO_URL = "https://github.com/<your-username>/hyperbolic-retinal-denoising.git"   # <-- edit once
BRANCH = "main"

In [ ]:
# Fail fast if the dataset is not attached (Add Input -> andrewmvd/retinal-disease-classification)
from pathlib import Path
hits = sorted(Path("/kaggle/input").glob("**/Training_Set"))
assert hits, "RFMiD NOT ATTACHED: Add Input -> 'retinal-disease-classification' (andrewmvd), then rerun."
print("RFMiD found at:", hits[0].parent)

In [ ]:
import os, subprocess, sys
REPO = "/kaggle/working/repo"
if not os.path.isdir(REPO):
    subprocess.check_call(["git", "clone", "-q", "--depth", "1", "-b", BRANCH, REPO_URL, REPO])
os.chdir(REPO)
print(subprocess.check_output(["git", "log", "-1", "--format=commit %h  %s"]).decode())
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "--no-deps", "-e", "third_party/hypll", "-e", "."])
subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", "lpips==0.1.4", "torchmetrics>=1.4", "pytest"])

## 1. Sanity checks (≈1 min) — stop here if anything fails

In [ ]:
!python -m pytest tests third_party/hypll/tests/manifolds/lorentz -q

## 2. Train + evaluate

In [ ]:
OUT = "/kaggle/working/results"
for seed in SEEDS:
    !python -m hypdenoise.train --config configs/{EXPERIMENT}.yaml --seed {seed} --set output_dir={OUT}
    if _exit_code != 0:
        raise RuntimeError(f"Training failed for seed {seed} - see the traceback above")
    !python -m hypdenoise.evaluate --run {OUT}/{EXPERIMENT}_seed{seed}
    if _exit_code != 0:
        raise RuntimeError(f"Evaluation failed for seed {seed} - see the traceback above")

## 3. Download
Everything is in `/kaggle/working/results/<experiment>_seed<k>/`:
`train_summary.json`, `history.json`, `test_summary.json`, `test_per_image.csv`, `best.pt`.
Copy the whole folder into `results/` of the repo on your laptop, then run
`python -m hypdenoise.compare --results results --reference euclidean`.

In [ ]:
!ls -la /kaggle/working/results/*